# Market Risk VaR and Expected Shortfall V1

This notebook builds an entry-level market risk prototype for a simple equity portfolio. It estimates one-day Value-at-Risk (VaR) and Expected Shortfall (ES) using historical simulation and parametric methods, then backtests VaR breaches against realized portfolio returns.

This is a learning prototype, not a production risk engine.

## 1. Imports and Configuration

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TICKERS = ['SPY', 'QQQ', 'JPM', 'AAPL', 'MSFT']
WEIGHTS = pd.Series({'SPY': 0.30, 'QQQ': 0.25, 'JPM': 0.15, 'AAPL': 0.15, 'MSFT': 0.15})
START_DATE = '2018-01-01'
WINDOW = 252
CONFIDENCE_LEVEL = 0.99

## 2. Download Prices and Build Portfolio Returns

In [ ]:
prices = yf.download(TICKERS, start=START_DATE, auto_adjust=True, progress=False)['Close']
returns = prices.pct_change().dropna()
portfolio_returns = returns[TICKERS].mul(WEIGHTS[TICKERS], axis=1).sum(axis=1).rename('portfolio_return')

prices.to_csv(RAW_DIR / 'market_risk_prices.csv')
portfolio_returns.to_csv(PROCESSED_DIR / 'market_risk_portfolio_returns_v1.csv')

print(prices.shape)
display(portfolio_returns.describe())

plt.figure(figsize=(10, 4))
(1 + portfolio_returns).cumprod().plot()
plt.title('Cumulative Portfolio Value')
plt.ylabel('Growth of $1')
plt.show()

## 3. Rolling VaR and Expected Shortfall

In [ ]:
def historical_var(window_returns: pd.Series, confidence: float) -> float:
    return -np.quantile(window_returns, 1 - confidence)


def historical_es(window_returns: pd.Series, confidence: float) -> float:
    cutoff = np.quantile(window_returns, 1 - confidence)
    return -window_returns[window_returns <= cutoff].mean()


def parametric_var(window_returns: pd.Series, confidence: float) -> float:
    mu = window_returns.mean()
    sigma = window_returns.std()
    return -(mu + sigma * norm.ppf(1 - confidence))


def parametric_es(window_returns: pd.Series, confidence: float) -> float:
    mu = window_returns.mean()
    sigma = window_returns.std()
    alpha = 1 - confidence
    return -(mu - sigma * norm.pdf(norm.ppf(alpha)) / alpha)


risk_rows = []
for i in range(WINDOW, len(portfolio_returns)):
    date = portfolio_returns.index[i]
    hist = portfolio_returns.iloc[i - WINDOW:i]
    realized = portfolio_returns.iloc[i]
    risk_rows.append({
        'date': date,
        'realized_return': realized,
        'historical_var': historical_var(hist, CONFIDENCE_LEVEL),
        'historical_es': historical_es(hist, CONFIDENCE_LEVEL),
        'parametric_var': parametric_var(hist, CONFIDENCE_LEVEL),
        'parametric_es': parametric_es(hist, CONFIDENCE_LEVEL),
    })

risk = pd.DataFrame(risk_rows).set_index('date')
risk['historical_breach'] = risk['realized_return'] < -risk['historical_var']
risk['parametric_breach'] = risk['realized_return'] < -risk['parametric_var']
risk.to_csv(PROCESSED_DIR / 'market_risk_var_es_v1.csv')
risk.head()

## 4. Backtesting

In [ ]:
expected_breach_rate = 1 - CONFIDENCE_LEVEL
backtest = pd.DataFrame({
    'method': ['historical', 'parametric'],
    'breach_count': [risk['historical_breach'].sum(), risk['parametric_breach'].sum()],
    'observations': [len(risk), len(risk)],
    'breach_rate': [risk['historical_breach'].mean(), risk['parametric_breach'].mean()],
    'expected_breach_rate': [expected_breach_rate, expected_breach_rate],
    'average_var': [risk['historical_var'].mean(), risk['parametric_var'].mean()],
    'average_es': [risk['historical_es'].mean(), risk['parametric_es'].mean()],
})
display(backtest)

plt.figure(figsize=(10, 5))
plt.plot(risk.index, risk['realized_return'], label='realized return', alpha=0.6)
plt.plot(risk.index, -risk['historical_var'], label='historical VaR threshold', color='red')
plt.scatter(risk.index[risk['historical_breach']], risk.loc[risk['historical_breach'], 'realized_return'], color='black', s=18, label='VaR breach')
plt.title('Historical Simulation VaR Backtest')
plt.ylabel('Daily return')
plt.legend()
plt.show()

## 5. Stress Period View

In [ ]:
worst_days = risk.nsmallest(10, 'realized_return')[['realized_return', 'historical_var', 'historical_es', 'parametric_var', 'parametric_es']]
display(worst_days)

rolling_vol = portfolio_returns.rolling(30).std() * np.sqrt(252)
plt.figure(figsize=(10, 4))
rolling_vol.plot()
plt.title('Rolling 30-Day Annualized Portfolio Volatility')
plt.ylabel('Annualized volatility')
plt.show()

## 6. Risk Modelling Notes

- VaR estimates a loss threshold at a confidence level.
- Expected Shortfall estimates the average loss beyond the VaR threshold.
- Backtesting checks whether realized losses breach VaR too often or too rarely.
- Parametric VaR assumes a normal return distribution, which can underestimate tail risk when returns are fat-tailed.
- Production market risk systems require clean market data, position data, stress testing, governance, and independent validation.